# Compositional design of a two-input transcriptional AND circuit

This notebook treats each candidate $D_i$ as a biological implementation, not
as a label for an arbitrarily selected asymptotic regime. Two characterized
Marionette sensors provide promoter activities $(\xi_1,\xi_2)$ in relative
promoter units (RPU). Lumped expression modules produce two binding partners,
which form a catalytic complex and generate one output transcription flux.

The model is
\[
\Sigma_1:\xi_1\mapsto Y_1,\qquad
\Sigma_2:\xi_2\mapsto Y_2,
\]
\[
Y_1+Y_2\rightleftharpoons C_3,
\qquad C_3\longrightarrow C_3+P,
\qquad y=\dot P=k_{\rm out}C_3.
\]
At quasi-steady binding,
\[
K C_3=(Y_{1,\rm tot}-C_3)(Y_{2,\rm tot}-C_3).
\]
The product-forming step is coarse grained: transcriptional resources are
implicit and neither input component is asserted to be chemically converted
into the reporter product.

In [ ]:
import math
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".matplotlib"))

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
import numpy as np
import pandas as pd

from compositional_regime_contracts import (
    CORES,
    DEFAULT_ON_THRESHOLD_NM_PER_MIN,
    DEFAULT_OFF_THRESHOLD_NM_PER_MIN,
    MARIONETTE_SENSORS,
    REDESIGNS,
    REDESIGN_PARAMETERS,
    complex_concentration,
    condition_flux_interval,
    infer_regime,
    make_candidate_designs,
    make_top_level_conditions,
    parameter_range_flux_bounds,
    redesigned_condition_flux_interval,
    redesigned_parameter_range_flux_bounds,
    sensor_activity,
    summarize_candidate_envelopes,
    summarize_redesigned_envelopes,
    verify_condition,
    verify_redesigned_condition,
)

pd.set_option("display.max_colwidth", 80)
candidates = make_candidate_designs()

## Characterized sensor interfaces

The aTc and Cuma entries below are transcribed from the Marionette sensor table
used in the Pacti case study, based on Meyer *et al.* (2019). Their physical
inducer concentrations are different, but each sensor exposes the same circuit
interface: output-promoter activity in RPU. No coordinate normalization by an
operating point or kinetic constant is performed.

In [ ]:
sensor_rows = []
for sensor in MARIONETTE_SENSORS.values():
    sensor_rows.append({
        "Sensor": sensor.name,
        "Promoter": sensor.promoter,
        "Regulator": sensor.regulator,
        "leak [RPU]": sensor.leak_rpu,
        "K [uM]": sensor.half_max_uM,
        "Hill n": sensor.hill_n,
        "measured max [RPU]": sensor.max_rpu,
        "validated linear max [RPU]": sensor.linear_max_rpu,
    })
sensor_table = pd.DataFrame(sensor_rows)
sensor_table

## Integration cores and initial design parameters

The three architectures use the effective affinities, promoter-to-component
gains, and output rates below as the fixed parameter values defining the
initial designs in the computational example.

In [ ]:
core_rows = []
for core in CORES:
    core_rows.append({
        "Core": core.name,
        "Input 1": core.component_1,
        "Input 2": core.component_2,
        "Output promoter": core.output_promoter,
        "K [nM]": core.affinity_nM,
        "k_out [1/min]": core.output_rate_per_min,
        "g1 [nM/RPU]": core.gain_1_nM_per_rpu,
        "g2 [nM/RPU]": core.gain_2_nM_per_rpu,
        "center regime": core.nominal_regime,
    })
core_table = pd.DataFrame(core_rows)
display(core_table)

uncertainty_table = pd.DataFrame([
    {"Parameter class": "lumped expression gains", "Range": f"nominal / {DEFAULT_UNCERTAINTY['gain_fold']} to nominal x {DEFAULT_UNCERTAINTY['gain_fold']}"},
    {"Parameter class": "effective affinity", "Range": f"nominal / {DEFAULT_UNCERTAINTY['affinity_fold']} to nominal x {DEFAULT_UNCERTAINTY['affinity_fold']}"},
    {"Parameter class": "output rate", "Range": f"nominal / {DEFAULT_UNCERTAINTY['output_rate_fold']} to nominal x {DEFAULT_UNCERTAINTY['output_rate_fold']}"},
])
display(uncertainty_table)

## Candidate designs

Each integration core is paired with both ordered assignments of aTc and Cuma.
Ordering changes the physical inducer-to-promoter map even when two candidates
have the same contract after the RPU interface. This is an intended example of
interface abstraction: downstream verification can be reused while the
physical sensor implementation changes.

In [ ]:
design_rows = []
for candidate in candidates:
    design_rows.append({
        "Design": candidate.name,
        "Core": candidate.core.name,
        "Sensor -> input 1": f"{candidate.sensor_1.name} -> {candidate.core.component_1}",
        "Sensor -> input 2": f"{candidate.sensor_2.name} -> {candidate.core.component_2}",
        "Regime at xi=(0.5,0.5)": infer_regime(candidate, 0.5, 0.5),
    })
design_table = pd.DataFrame(design_rows)
design_table

## Top-level AND-gate specification

The desired AND behavior contains one ON region and two OFF regions:
\[
(\xi_1,\xi_2)\in B_{\rm ON}\Longrightarrow y\ge y_{\rm ON},
\]
\[
\xi_1\in B_{\rm OFF}\ \text{or}\ \xi_2\in B_{\rm OFF}
\Longrightarrow y\le y_{\rm OFF}.
\]
The input regions are stated in promoter activity units. The ON region lies inside
the experimentally useful Marionette RPU range. The OFF interval begins near
the characterized leak floors and extends to 20% of the ON onset; the interval
from 0.07 to 0.35 RPU is deliberately left unconstrained. The
round exploratory flux targets impose a twofold output separation. In an
experimental design, these levels would be set from measured sensor variation
and the activation and burden limits of the downstream receiver.

In [ ]:
conditions = make_top_level_conditions()
envelope_summary = summarize_candidate_envelopes(candidates, conditions)
condition_table = pd.DataFrame([
    {
        "Condition": condition.name,
        "xi1 [RPU]": condition.xi1_bounds,
        "xi2 [RPU]": condition.xi2_bounds,
        "Required flux": (
            f">= {condition.flux_threshold:.2f} nM/min"
            if condition.guarantee_kind == "lower"
            else f"<= {condition.flux_threshold:.2f} nM/min"
        ),
    }
    for condition in conditions
])
display(pd.DataFrame({
    "Core": envelope_summary["on_floors"].keys(),
    "worst-case ON floor [nM/min]": envelope_summary["on_floors"].values(),
    "worst-case OFF ceiling [nM/min]": envelope_summary["off_ceilings"].values(),
}))
display(condition_table)

## Composition and formal checks

For a finite operating region, the catalytic-core contract uses the globally valid
bounds
\[
\frac{Y_1Y_2}{K+Y_1+Y_2}\le C_3
\le\min\left\{Y_1,Y_2,\frac{Y_1Y_2}{K}\right\}.
\]
Monotonic endpoint bounds tighten this contract over each region. The fixed
initial design parameters are used when composing the two lumped expression
contracts with the core contract. A design implements the complete AND
specification only if it refines the ON, OFF-1, and OFF-2 contracts.

The top-level object is therefore the mode-indexed family
\[
\mathfrak C_{\rm spec}=\{\mathcal C_{\rm ON},
\mathcal C_{\rm OFF,1},\mathcal C_{\rm OFF,2}\},
\qquad
D\models\mathfrak C_{\rm spec}
\Longleftrightarrow
\bigwedge_q \mathcal C_D^q\preceq\mathcal C_{\rm spec}^q.
\]
An ordinary Pacti merge is not used: it would conjoin the mutually exclusive
mode assumptions instead of preserving the three conditional requirements.

In [ ]:
verification_rows = []
verification = {}
for candidate in candidates:
    condition_results = {}
    for condition in conditions:
        result = verify_condition(candidate, condition)
        condition_results[condition.name] = result
    verification[candidate.name] = condition_results
    verification_rows.append({
        "Design": candidate.name,
        "Core": candidate.core.name,
        "Sensors (input 1, input 2)": f"({candidate.sensor_1.name}, {candidate.sensor_2.name})",
        "A_ON subset A_D": condition_results["ON"]["assumptions"],
        "G_D implies G_ON": condition_results["ON"]["guarantees"],
        "A_OFF,1 subset A_D": condition_results["OFF-1"]["assumptions"],
        "G_D implies G_OFF,1": condition_results["OFF-1"]["guarantees"],
        "A_OFF,2 subset A_D": condition_results["OFF-2"]["assumptions"],
        "G_D implies G_OFF,2": condition_results["OFF-2"]["guarantees"],
        "Full AND refinement": all(result["refines"] for result in condition_results.values()),
    })
verification_table = pd.DataFrame(verification_rows)
verification_table

In [ ]:
interval_rows = []
for candidate in candidates[::2]:
    row = {"Core": candidate.core.name}
    for condition in conditions:
        lo, hi = condition_flux_interval(
            candidate, condition.xi1_bounds, condition.xi2_bounds
        )
        row[condition.name] = f"[{lo:.1f}, {hi:.1f}]"
    interval_rows.append(row)
pd.DataFrame(interval_rows)

### Initial design insight

The broadened LOW range exposes a limitation that the leak-only specification
hid: none of the three direct complex-formation cores satisfies the complete
AND contract. HrpR/HrpS and InvF/SicA reach the ON target but exceed the OFF
ceiling; split T7 RNAP also falls slightly below the ON target. The equality of
each ordered pair after the RPU interface still shows what the abstraction
provides; their required inducer concentrations remain different in the sensor
contracts.

## Contract-guided redesigns

The fixed ON/OFF specification remains unchanged. Each initial design is
evaluated at its stated parameter values, and each redesign updates the
mechanism-specific design parameter(s). The contract calculation determines
whether the resulting design satisfies all three modes.

1. **HrpR/HrpS:** treat the HrpV inhibitory level as a tunable regulation
   parameter. For the fixed specification, the feasible singleton-design
   interval is approximately $0.114$--$0.131\,\mu\mathrm{M}$; use
   $V_{\mathrm{eff}}=0.12\,\mu\mathrm{M}$.
2. **split T7 RNAP:** use the effective fragment-association parameter as the
   redesign variable. The full contract is feasible for approximately
   $K=2.286$--$3.165\,\mu\mathrm{M}$; use $K=3.0\,\mu\mathrm{M}$ to
   increase OFF-state margin while retaining the ON guarantee.
3. **InvF/SicA:** use the reduced SicA--SipC turnover model with the published
   $K_{YZ}=28\,\mathrm{nM}$, $k_Z=0.1\,\mathrm{min}^{-1}$, and degradation
   rates $\gamma_X=\gamma_{XY}=0.022$, $\gamma_Y=0.046$,
   $\gamma_Z=0.139$, and $\gamma_{YZ}=0.022\,\mathrm{min}^{-1}$.

The cells below compute the contract-feasible parameter windows for the first
two redesigns and then run the same mode-wise refinement checks for all three.

In [ ]:
# Contract-feasible fixed-parameter redesign windows for the two direct QSS cores.
representative_by_core = {}
for candidate in candidates:
    representative_by_core.setdefault(candidate.core.name, candidate)

hrp = representative_by_core["HrpR/HrpS"].core
c_on = float(complex_concentration(
    hrp.gain_1_nM_per_rpu * 0.35,
    hrp.gain_2_nM_per_rpu * 0.35,
    hrp.affinity_nM,
))
c_off1 = float(complex_concentration(
    hrp.gain_1_nM_per_rpu * 0.07,
    hrp.gain_2_nM_per_rpu * 0.65,
    hrp.affinity_nM,
))
c_off2 = float(complex_concentration(
    hrp.gain_1_nM_per_rpu * 0.65,
    hrp.gain_2_nM_per_rpu * 0.07,
    hrp.affinity_nM,
))
hrpv_low = max(
    c_off1 - DEFAULT_OFF_THRESHOLD_NM_PER_MIN / hrp.output_rate_per_min,
    c_off2 - DEFAULT_OFF_THRESHOLD_NM_PER_MIN / hrp.output_rate_per_min,
    0.0,
)
hrpv_high = c_on - DEFAULT_ON_THRESHOLD_NM_PER_MIN / hrp.output_rate_per_min

split = representative_by_core["split T7 RNAP"].core
def affinity_at_complex(input_1_nM, input_2_nM, complex_nM):
    return ((input_1_nM - complex_nM) * (input_2_nM - complex_nM)) / complex_nM

c_on_req = DEFAULT_ON_THRESHOLD_NM_PER_MIN / split.output_rate_per_min
c_off_req = DEFAULT_OFF_THRESHOLD_NM_PER_MIN / split.output_rate_per_min
split_k_high = affinity_at_complex(
    split.gain_1_nM_per_rpu * 0.35,
    split.gain_2_nM_per_rpu * 0.35,
    c_on_req,
)
split_k_low = max(
    affinity_at_complex(
        split.gain_1_nM_per_rpu * 0.07,
        split.gain_2_nM_per_rpu * 0.65,
        c_off_req,
    ),
    affinity_at_complex(
        split.gain_1_nM_per_rpu * 0.65,
        split.gain_2_nM_per_rpu * 0.07,
        c_off_req,
    ),
)

pd.DataFrame([
    {
        "Core": "HrpR/HrpS",
        "design variable": "HrpV [nM]",
        "feasible low": hrpv_low,
        "feasible high": hrpv_high,
        "selected": REDESIGN_PARAMETERS["HrpR/HrpS"]["hrpv_nM"],
    },
    {
        "Core": "split T7 RNAP",
        "design variable": "effective K [nM]",
        "feasible low": split_k_low,
        "feasible high": split_k_high,
        "selected": REDESIGN_PARAMETERS["split T7 RNAP"]["affinity_nM"],
    },
])

In [ ]:
redesign_rows = []
for core_name, redesign in REDESIGNS.items():
    params = REDESIGN_PARAMETERS[core_name]
    redesign_rows.append({
        "Core": core_name,
        "Redesign": redesign.label,
        "Mechanism": redesign.mechanism,
        "Selected redesign parameters": str(params),
        "Source": redesign.source,
    })
display(pd.DataFrame(redesign_rows))

display(pd.DataFrame([{
    "Redesign contract range": "fixed selected synthesis point",
    "gain fold": REDESIGN_UNCERTAINTY["gain_fold"],
    "affinity fold": REDESIGN_UNCERTAINTY["affinity_fold"],
    "output-rate fold": REDESIGN_UNCERTAINTY["output_rate_fold"],
}]))

In [ ]:
redesigned_envelope_summary = summarize_redesigned_envelopes(candidates, conditions)

redesigned_verification_rows = []
redesigned_verification = {}
for candidate in candidates:
    condition_results = {}
    for condition in conditions:
        result = verify_redesigned_condition(candidate, condition)
        condition_results[condition.name] = result
    redesigned_verification[candidate.name] = condition_results
    redesigned_verification_rows.append({
        "Design": candidate.name,
        "Core": candidate.core.name,
        "Redesign": REDESIGNS[candidate.core.name].label,
        "Sensors (input 1, input 2)": f"({candidate.sensor_1.name}, {candidate.sensor_2.name})",
        "ON": condition_results["ON"]["refines"],
        "OFF-1": condition_results["OFF-1"]["refines"],
        "OFF-2": condition_results["OFF-2"]["refines"],
        "Full AND refinement": all(result["refines"] for result in condition_results.values()),
    })
redesigned_verification_table = pd.DataFrame(redesigned_verification_rows)
display(redesigned_verification_table)

redesigned_interval_rows = []
for candidate in candidates[::2]:
    row = {
        "Core": candidate.core.name,
        "Redesign": REDESIGNS[candidate.core.name].label,
    }
    for condition in conditions:
        lo, hi = redesigned_condition_flux_interval(
            candidate, condition.xi1_bounds, condition.xi2_bounds
        )
        row[condition.name] = f"[{lo:.3g}, {hi:.3g}]"
    redesigned_interval_rows.append(row)
pd.DataFrame(redesigned_interval_rows)